# Reintegration Readiness (90-day)

## 1. Problem Framing
- **Business question:** Which residents are likely to reach reintegration readiness in the next 90 days?
- **Predictive use:** Prioritize support resources.
- **Explanatory use:** Identify factors associated with readiness progression.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score

DATA = Path('lighthouse_csv_v7')
res = pd.read_csv(DATA / 'residents.csv', parse_dates=['date_enrolled','date_closed'])
edu = pd.read_csv(DATA / 'education_records.csv', parse_dates=['record_date'])
health = pd.read_csv(DATA / 'health_wellbeing_records.csv', parse_dates=['record_date'])
proc = pd.read_csv(DATA / 'process_recordings.csv', parse_dates=['session_date'])
vis = pd.read_csv(DATA / 'home_visitations.csv', parse_dates=['visit_date'])

# Leakage checklist:
# 1) Anchor at enrollment.
# 2) Features from first 90 days.
# 3) Target from future closure window only (up to 365 days).
base = res[['resident_id','reintegration_type','case_category','present_age','date_enrolled','date_closed']].dropna(subset=['date_enrolled']).copy()
base['feat_end'] = base['date_enrolled'] + pd.Timedelta(days=90)
base['target_ready_365d'] = ((base['date_closed'].notna()) & (base['date_closed'] > base['feat_end']) & (base['date_closed'] <= base['feat_end'] + pd.Timedelta(days=365))).astype(int)

edu_m = base[['resident_id','date_enrolled','feat_end']].merge(edu, on='resident_id', how='left')
edu_m = edu_m[(edu_m['record_date'] >= edu_m['date_enrolled']) & (edu_m['record_date'] <= edu_m['feat_end'])]
edu_agg = edu_m.groupby('resident_id').agg(edu_progress_90d=('progress_percent','mean'),edu_attendance_90d=('attendance_rate','mean')).reset_index()

health_m = base[['resident_id','date_enrolled','feat_end']].merge(health, on='resident_id', how='left')
health_m = health_m[(health_m['record_date'] >= health_m['date_enrolled']) & (health_m['record_date'] <= health_m['feat_end'])]
health_agg = health_m.groupby('resident_id').agg(health_avg_90d=('general_health_score','mean')).reset_index()

proc_m = base[['resident_id','date_enrolled','feat_end']].merge(proc, on='resident_id', how='left')
proc_m = proc_m[(proc_m['session_date'] >= proc_m['date_enrolled']) & (proc_m['session_date'] <= proc_m['feat_end'])]
proc_agg = proc_m.groupby('resident_id').agg(session_count_90d=('recording_id','count'),progress_rate_90d=('progress_noted','mean')).reset_index()

vis_m = base[['resident_id','date_enrolled','feat_end']].merge(vis, on='resident_id', how='left')
vis_m = vis_m[(vis_m['visit_date'] >= vis_m['date_enrolled']) & (vis_m['visit_date'] <= vis_m['feat_end'])]
vis_agg = vis_m.groupby('resident_id').agg(family_supportive_90d=('family_cooperation_level', lambda s: (s == 'Supportive').mean())).reset_index()

df = (base[['resident_id','reintegration_type','case_category','present_age','date_enrolled','target_ready_365d']]
      .merge(edu_agg, on='resident_id', how='left')
      .merge(health_agg, on='resident_id', how='left')
      .merge(proc_agg, on='resident_id', how='left')
      .merge(vis_agg, on='resident_id', how='left'))

# Fill numerics and categoricals separately to keep encoder input types consistent.
num_fill_cols = [c for c in df.columns if c not in ['reintegration_type','case_category','date_enrolled']]
df[num_fill_cols] = df[num_fill_cols].fillna(0)
df['reintegration_type'] = df['reintegration_type'].fillna('Unknown')
df['case_category'] = df['case_category'].fillna('Unknown')

df = df.sort_values('date_enrolled')
split_date = df['date_enrolled'].quantile(0.8)
train_df = df[df['date_enrolled'] < split_date].copy()
test_df = df[df['date_enrolled'] >= split_date].copy()

X_train = train_df.drop(columns=['resident_id','date_enrolled','target_ready_365d']).copy()
y_train = train_df['target_ready_365d']
X_test = test_df.drop(columns=['resident_id','date_enrolled','target_ready_365d']).copy()
y_test = test_df['target_ready_365d']

X = df.drop(columns=['resident_id','date_enrolled','target_ready_365d']).copy()
y = df['target_ready_365d']

num_cols = X_train.select_dtypes(include=['number']).columns.tolist()
cat_cols = [c for c in X_train.columns if c not in num_cols]
for c in cat_cols:
    X_train[c] = X_train[c].astype('object').fillna('Unknown')
    X_test[c] = X_test[c].astype('object').fillna('Unknown')
    X[c] = X[c].astype('object').fillna('Unknown')

pre = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')),('sc', StandardScaler())]), num_cols),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')),('oh', OneHotEncoder(handle_unknown='ignore'))]), cat_cols)
])

baseline_p = np.zeros(len(y_test), dtype=int)
print('Baseline_AlwaysNegative', 'F1', round(f1_score(y_test, baseline_p, zero_division=0), 3), 'Recall', round(recall_score(y_test, baseline_p, zero_division=0), 3))

exp_model = Pipeline([('pre', pre), ('clf', LogisticRegression(max_iter=2000, class_weight='balanced'))])
pred_model = Pipeline([('pre', pre), ('clf', GradientBoostingClassifier(random_state=42))])

val_cut = train_df['date_enrolled'].quantile(0.8)
tr_df = train_df[train_df['date_enrolled'] < val_cut]
va_df = train_df[train_df['date_enrolled'] >= val_cut]
X_tr = tr_df.drop(columns=['resident_id','date_enrolled','target_ready_365d'])
y_tr = tr_df['target_ready_365d']
X_va = va_df.drop(columns=['resident_id','date_enrolled','target_ready_365d'])
y_va = va_df['target_ready_365d']
for c in cat_cols:
    X_tr[c] = X_tr[c].astype('object').fillna('Unknown')
    X_va[c] = X_va[c].astype('object').fillna('Unknown')

def best_threshold(y_true, score):
    cand = np.linspace(0.2, 0.8, 13)
    vals = [(t, f1_score(y_true, (score >= t).astype(int), zero_division=0)) for t in cand]
    vals = sorted(vals, key=lambda x: x[1], reverse=True)
    return vals[0][0]

for name, model in [('Explanatory_LogReg', exp_model), ('Predictive_GB', pred_model)]:
    model.fit(X_tr, y_tr)
    s_val = model.predict_proba(X_va)[:,1]
    t_star = best_threshold(y_va, s_val)

    model.fit(X_train, y_train)
    s = model.predict_proba(X_test)[:,1]
    p = (s >= t_star).astype(int)
    print(name, 'threshold', round(float(t_star), 2), 'AUC', round(roc_auc_score(y_test, s), 3), 'F1', round(f1_score(y_test, p, zero_division=0), 3),
          'Precision', round(precision_score(y_test, p, zero_division=0), 3), 'Recall', round(recall_score(y_test, p, zero_division=0), 3))


## 4. Evaluation & Interpretation
- High recall is useful to avoid missing residents who could benefit from reintegration support.
- Precision controls staff workload by reducing false alarms.

## 5. Causal and Relationship Analysis
- Typical positive associations: strong attendance/progress, supportive family cooperation, improved emotional progression.
- Observational limitation remains: association does not imply causation.

## 6. Deployment Notes
- Endpoint: `POST /api/ml/reintegration-readiness`.
- UI: case conference prioritization queue and readiness trend widget.

In [ ]:
# Model selection: compare multiple classification algorithms with adaptive CV
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

candidate_models = {
    'LogisticRegression': LogisticRegression(max_iter=2000, class_weight='balanced'),
    'DecisionTree': DecisionTreeClassifier(random_state=42, class_weight='balanced'),
    'RandomForest': RandomForestClassifier(n_estimators=300, random_state=42, class_weight='balanced'),
    'GradientBoosting': GradientBoostingClassifier(random_state=42)
}

min_class_count = int(pd.Series(y).value_counts().min())
n_splits = max(2, min(5, min_class_count))
cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
scoring = {
    'auc': 'roc_auc',
    'f1': 'f1',
    'precision': 'precision',
    'recall': 'recall'
}

rows = []
for model_name, estimator in candidate_models.items():
    pipe = Pipeline([('pre', pre), ('clf', estimator)])
    scores = cross_validate(pipe, X, y, cv=cv, scoring=scoring, n_jobs=-1, error_score='raise')
    rows.append({
        'model': model_name,
        'cv_folds': n_splits,
        'auc_mean': scores['test_auc'].mean(),
        'f1_mean': scores['test_f1'].mean(),
        'precision_mean': scores['test_precision'].mean(),
        'recall_mean': scores['test_recall'].mean()
    })

model_selection_results = pd.DataFrame(rows).sort_values('auc_mean', ascending=False)
print('Model selection results (Reintegration Readiness):')
print(model_selection_results.to_string(index=False))